In [5]:
import os
import evaluate as ev

WORK = "annotation_work"      # the --out directory the server wrote to
DECK = "09-visual-design-wo-tufte"            # deck_id (the prefix on the json filenames)
GT   = "alice"                   # your annotator name (the ground truth)
PARTICIPANTS = ["haiku4d5-base", "sonnet5med-think", "sonnet5med-think-full"]

def rel_file(who):    return os.path.join(WORK, f"{DECK}.relationships.{who}.json")
def style_file(who):  return os.path.join(WORK, f"{DECK}.styles.{who}.json")

gt_rels   = ev.load(rel_file(GT))
gt_styles = ev.load_styles(style_file(GT)) if os.path.exists(style_file(GT)) else ev.derive_styles(gt_rels)


In [6]:
# Per-participant detailed report
reports = {}
for who in PARTICIPANTS:
    pred_rels = ev.load(rel_file(who))
    sf = style_file(who)
    pred_styles = ev.load_styles(sf) if os.path.exists(sf) else ev.derive_styles(pred_rels)

    rel_report = ev.score_relationships(gt_rels, pred_rels)
    style_report = ev.score_styles(gt_styles, pred_styles, gt_rels, pred_rels)
    reports[who] = (rel_report, style_report)

    ev.print_report(rel_report, style_report, label=who)
    print()

=== haiku4d5-base ===
relationships (per-attribute co-membership F1)
  attr              P      R     F1   (tp/pred/gt)
  font_color     0.22   0.35   0.27   (5222/23492/14814)
  y              0.14   0.26   0.18   (2383/16687/9204)
  text           0.09   0.14   0.11   (915/9963/6690)
  x              0.10   0.06   0.07   (345/3321/6142)
  fill           1.00   0.00   0.00   (0/0/887)
  h              1.00   0.00   0.00   (0/0/29955)
  img_content    1.00   0.00   0.00   (0/0/16)
  line_color     1.00   0.00   0.00   (0/0/10)
  w              1.00   0.00   0.00   (0/0/9966)
  MACRO          0.62   0.09   0.07
  MICRO          0.17   0.11   0.14
styles (attribute x object cell overlap, best-match Jaccard)
  P 0.12  R 0.02  F1 0.04   (gt=32, pred=4)

=== sonnet5med-think ===
relationships (per-attribute co-membership F1)
  attr              P      R     F1   (tp/pred/gt)
  text           1.00   1.00   1.00   (6690/6715/6690)
  img_content    0.94   1.00   0.97   (16/17/16)
  h          

In [7]:
print(f"{'participant':<14} {'macroF1':>8} {'microF1':>8} {'styleF1':>8}")
for who, (rel_report, style_report) in reports.items():
    print(f"{who:<14} {rel_report['macro']['f1']:8.2f} "
          f"{rel_report['micro']['f1']:8.2f} {style_report['f1']:8.2f}")

participant     macroF1  microF1  styleF1
haiku4d5-base      0.07     0.14     0.04
sonnet5med-think     0.65     0.81     0.21
sonnet5med-think-full     0.76     0.79     0.08


In [8]:
# The per-attribute dict carries the raw counts (tp / predicted-pairs / gt-pairs),
# so you can see exactly where a participant diverged.
who = PARTICIPANTS[0]
for attr, v in sorted(reports[who][0]["per_attr"].items()):
    print(f"{attr:<12} F1={v['f1']:.2f}  tp={v['tp']} pred={v['pred']} gt={v['gt']}")


fill         F1=0.00  tp=0 pred=0 gt=887
font_color   F1=0.27  tp=5222 pred=23492 gt=14814
h            F1=0.00  tp=0 pred=0 gt=29955
img_content  F1=0.00  tp=0 pred=0 gt=16
line_color   F1=0.00  tp=0 pred=0 gt=10
text         F1=0.11  tp=915 pred=9963 gt=6690
w            F1=0.00  tp=0 pred=0 gt=9966
x            F1=0.07  tp=345 pred=3321 gt=6142
y            F1=0.18  tp=2383 pred=16687 gt=9204
